# Fine-tuning `granite-speech-5.0-470m-turboctc`

Fine-tunes IBM's [470M CTC speech model](https://huggingface.co/ibm-granite/granite-speech-5.0-470m-turboctc)
on [FLEURS German](https://huggingface.co/datasets/google/fleurs) (~10 h, ungated).

The model is English-only, so German is a real domain shift. Verified end to end on one H100
(~37 min, defaults below unchanged):

| | baseline | after 8 epochs |
|---|---|---|
| WER | 77.5% | **51.9%** |
| CER | 36.5% | **20.0%** |

To use your own data, change one cell (§2).

This is a pure CTC encoder — no LLM decoder, no chat template, no LoRA. Audio in, text out, one
non-autoregressive forward pass.

**Runtime → Change runtime type → GPU** before running.

## 1. Setup

Needs `transformers >= 5.16`, newer than Colab ships.

In [ ]:
%%capture
# transformers >= 5.16 is required and newer than Colab ships.
!pip install -q -U git+https://github.com/huggingface/transformers.git
!pip install -q -U "datasets>=4.0" accelerate evaluate jiwer

In [ ]:
# `datasets` decodes audio only through torchcodec, which is ABI-linked to torch's
# CUDA runtime but declares no torch dependency -- so a plain `pip install -U
# torchcodec` can grab a build for a different CUDA and every decode then dies with
#   OSError: libnvrtc.so.NN: cannot open shared object file
# Reinstalling the same version does not help; the version has to match torch.
import subprocess, sys

import torch

TORCHCODEC_FOR_TORCH = {   # torch minor -> torchcodec release built against it
    "2.9": "0.9.1", "2.10": "0.10.0", "2.11": "0.11.1",
    "2.12": "0.12.0", "2.13": "0.15.0", "2.14": "0.16.0",
}
torch_mm = ".".join(torch.__version__.split(".")[:2])
pin = TORCHCODEC_FOR_TORCH.get(torch_mm)
print(f"torch {torch.__version__} -> torchcodec {pin or 'unknown (leaving as-is)'}")

if pin:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "--force-reinstall", "--no-deps", f"torchcodec=={pin}"], check=True)
    print("installed; now Runtime > Restart session, then continue from the next cell.")

*Restart the session once after installing, then run from here.* The next cell verifies the
environment — if audio decoding is broken it says so here rather than mid-training.

In [ ]:
# Import datasets FIRST: it initializes torchcodec's decoder, which must load its
# libraries before torch claims them.
from datasets import Audio, Dataset, Features, Value, load_dataset   # noqa: F401
import torch, transformers
from packaging.version import parse as V

assert V(transformers.__version__).release >= (5, 16), (
    f"need transformers>=5.16, got {transformers.__version__}; rerun the install cell "
    "then Runtime > Restart session")

# `datasets` decodes audio only through torchcodec, which is ABI-linked to torch.
# `pip install -U torchcodec` can pull a build for a different CUDA than the runtime
# has; the failure then surfaces as OSError: libnvrtc.so.NN on the first decode.
try:
    load_dataset("hf-internal-testing/librispeech_asr_dummy", "clean",
                 split="validation[:1]").cast_column("audio", Audio(sampling_rate=16000))[0]
except OSError as e:
    raise SystemExit(
        f"Audio decoding is broken ({e}).\ntorchcodec does not match this torch "
        "build. Re-run the torchcodec pin cell above (it picks the version for your "
        "torch), then Runtime > Restart session. Reinstalling the same version will "
        "not fix it.") from e

print(f"transformers {transformers.__version__} | torch {torch.__version__} "
      f"| cuda {torch.cuda.is_available()} | audio decode OK")
if not torch.cuda.is_available():
    print("WARNING: no GPU -- Runtime > Change runtime type > GPU")

In [ ]:
from datasets import Audio, Dataset, Features, Value, load_dataset   # import before torch
import numpy as np
import torch
from transformers import AutoProcessor, GraniteSpeech5ForCTC

MODEL_ID = "ibm-granite/granite-speech-5.0-470m-turboctc"

processor = AutoProcessor.from_pretrained(MODEL_ID)
model = GraniteSpeech5ForCTC.from_pretrained(MODEL_ID, dtype=torch.float32)

SAMPLING_RATE = processor.feature_extractor.sampling_rate
BLANK_ID = model.config.pad_token_id      # CTC blank == pad == 0

print(f"{model.num_parameters()/1e6:.0f}M params | {SAMPLING_RATE} Hz | blank id {BLANK_ID}")

## 2. Data

Any `Dataset` with these three columns works — swap `load_data()` for your own source:

| column | type |
|---|---|
| `audio` | `{"array": float32[...], "sampling_rate": 16000}` |
| `text` | `str` |
| `input_length` | duration in seconds |

In [ ]:
N_EVAL = 400
MAX_SECONDS = 16.0


def gen_rows():
    """Yield one clip at a time. FLEURS calls the transcript `transcription`."""
    for split in ("train", "validation"):
        stream = load_dataset(
            "google/fleurs", "de_de", split=split, streaming=True,
        ).cast_column("audio", Audio(sampling_rate=SAMPLING_RATE))
        for row in stream:
            audio = row["audio"]
            dur = len(audio["array"]) / audio["sampling_rate"]
            if dur <= MAX_SECONDS and row["transcription"].strip():
                yield {
                    "audio": {"array": np.asarray(audio["array"], dtype=np.float32),
                              "sampling_rate": audio["sampling_rate"]},
                    "text": row["transcription"],
                    "input_length": dur,
                }


# from_generator streams straight to an on-disk Arrow file, so only one clip is ever
# in RAM. Building a Python list first and calling Dataset.from_list() needs ~4 GB
# for this corpus (plus a full copy during the map below) and OOMs a free Colab.
full = Dataset.from_generator(gen_rows, features=Features({
    "audio": Audio(sampling_rate=SAMPLING_RATE),
    "text": Value("string"),
    "input_length": Value("float32"),
}))
assert {"audio", "text", "input_length"} <= set(full.column_names)

split = full.train_test_split(test_size=N_EVAL, seed=0)
train_raw, eval_raw = split["train"], split["test"]
print(f"train {len(train_raw)} | eval {len(eval_raw)}")
print(train_raw[0]["text"][:90])

### Normalize

Two separate jobs. **Targets** get only lowercasing and character filtering — `KEEP_CHARS` must
include your language's letters, since anything dropped here the model can never learn to emit.
**Scoring** uses Whisper's `BasicTextNormalizer` (use `EnglishTextNormalizer` only for English).

In [ ]:
import re

from transformers.models.whisper.english_normalizer import BasicTextNormalizer

KEEP_CHARS = r"a-zäöüß' "        # German; English would be r"a-z' "
_keep = re.compile(f"[^{KEEP_CHARS}]+")
_basic = BasicTextNormalizer()


def normalize(text):
    return re.sub(r"\s+", " ", _keep.sub(" ", text.lower().replace("-", " "))).strip()


def prepare(ds):
    drop = [c for c in ds.column_names if c not in ("audio", "text", "input_length")]
    # Two things matter here:
    #  * input_columns= -- without it `datasets` deserializes every clip's audio just
    #    to read the text (~15 rows/s vs ~11000).
    #  * writer_batch_size -- map() writes a new Arrow file; a small batch keeps the
    #    transient buffer tiny instead of holding ~2 GB of audio in RAM.
    ds = ds.map(lambda t: {"text": normalize(t)}, input_columns="text",
                remove_columns=drop, writer_batch_size=64)
    return ds.filter(lambda t: len(t) > 0, input_columns="text",
                     writer_batch_size=64)


train_ds, eval_ds = prepare(train_raw), prepare(eval_raw)
print(train_ds[0]["text"][:90])

### Check the CTC frame budget

The encoder emits **12.5 frames/s** and CTC needs `len(tokens) <= frames`. Violating clips get a
zeroed loss (`zero_infinity`) — they train on nothing while looking like a plateau, so drop them.

If this drops more than a few percent, check your sampling rate or your script: the tokenizer is
English BPE, costing ~0.23 tokens/char for German but **1.9–3.0 for Russian, Hindi and Japanese**,
which need 18–31 tokens/s and cannot be aligned on this model at all.

In [ ]:
fe = processor.feature_extractor
SUBSAMPLE = 2 ** len(model.config.encoder_config.subsample_layers)
tok = processor.tokenizer


def n_frames(seconds):
    mel = int(seconds * SAMPLING_RATE) // fe.hop_length
    return (-(-mel // fe.frame_stacking)) // SUBSAMPLE


def alignable(text, input_length):
    return 0 < len(tok(text, add_special_tokens=False)["input_ids"]) <= n_frames(input_length)


before = len(train_ds)
train_ds = train_ds.filter(alignable, input_columns=["text", "input_length"])
eval_ds = eval_ds.filter(alignable, input_columns=["text", "input_length"])
print(f"dropped {before - len(train_ds)}/{before} | train {len(train_ds)} eval {len(eval_ds)}")

## 3. Collator

One CTC-specific trap: pad labels with the **blank id (0)**, not `-100`. The model computes
`target_lengths = (labels != pad_token_id).sum(-1)` and never looks for `-100`, so `-100` padding is
silently counted as real target tokens.

In [ ]:
from dataclasses import dataclass


@dataclass
class CTCCollator:
    processor: object

    def __call__(self, features):
        batch = self.processor(
            [f["audio"]["array"] for f in features],
            text=[f["text"] for f in features],
            sampling_rate=SAMPLING_RATE, padding="longest", return_tensors="pt",
        )
        labels = batch["labels"]
        batch["labels"] = labels.masked_fill(
            labels == self.processor.tokenizer.pad_token_id, BLANK_ID)
        return batch


collator = CTCCollator(processor)
for k, v in collator([train_ds[i] for i in range(4)]).items():
    print(k, tuple(v.shape))

## 4. Metric

WER and CER from `generate()`. `Trainer` pads gathered batches with `-100`, which the tokenizer
cannot decode, so map those back to blank first.

In [ ]:
import evaluate

wer_metric, cer_metric = evaluate.load("wer"), evaluate.load("cer")


def compute_metrics(pred):
    ids = pred.predictions
    if isinstance(ids, tuple):
        ids = ids[0]
    ids = np.where(np.asarray(ids) < 0, BLANK_ID, ids)
    if ids.ndim == 3:
        ids = ids.argmax(-1)

    hyps = processor.batch_decode(ids, skip_special_tokens=True)
    refs = processor.batch_decode(
        np.where(np.asarray(pred.label_ids) < 0, BLANK_ID, pred.label_ids),
        skip_special_tokens=True)

    pairs = [(_basic(h), _basic(r)) for h, r in zip(hyps, refs)]
    pairs = [(h, r) for h, r in pairs if r]
    h, r = zip(*pairs)
    return {"wer": 100 * wer_metric.compute(predictions=list(h), references=list(r)),
            "cer": 100 * cer_metric.compute(predictions=list(h), references=list(r))}

## 5. Train

`Trainer` would evaluate on frame-level logits, which are full of blanks and repeats — so
`prediction_step` calls `generate()` to get the real greedy CTC decode.

**Gate on WER, not loss.** CTC loss and WER can move in opposite directions; `load_best_model_at_end`
with `metric_for_best_model="wer"` keeps the checkpoint that actually decoded best.

In [ ]:
from transformers import Trainer, TrainingArguments


class CTCTrainer(Trainer):
    def prediction_step(self, model, inputs, prediction_loss_only, ignore_keys=None):
        inputs = self._prepare_inputs(inputs)
        labels = inputs.get("labels")
        with torch.no_grad():
            loss = model(**inputs).loss
            if prediction_loss_only:
                return (loss, None, None)
            gen = model.generate(input_features=inputs["input_features"],
                                 attention_mask=inputs.get("attention_mask"))
        if gen.shape[-1] < labels.shape[-1]:
            gen = torch.nn.functional.pad(
                gen, (0, labels.shape[-1] - gen.shape[-1]), value=BLANK_ID)
        return (loss, gen, labels)


model.gradient_checkpointing_enable()

args = TrainingArguments(
    output_dir="granite-turboctc-de",
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=2,
    learning_rate=1e-5,
    warmup_steps=100,
    num_train_epochs=8,
    bf16=torch.cuda.is_bf16_supported(),
    fp16=not torch.cuda.is_bf16_supported(),
    gradient_checkpointing=True,
    train_sampling_strategy="group_by_length",
    length_column_name="input_length",
    logging_steps=25,
    eval_strategy="steps",
    eval_steps=50,
    save_strategy="steps",
    save_steps=50,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="wer",
    greater_is_better=False,
    remove_unused_columns=False,
    report_to="none",
)

trainer = CTCTrainer(
    model=model, args=args, train_dataset=train_ds, eval_dataset=eval_ds,
    data_collator=collator, processing_class=processor, compute_metrics=compute_metrics,
)

Baseline first — it tells you whether fine-tuning can help at all.

In [ ]:
print(trainer.evaluate(metric_key_prefix="base"))

In [ ]:
trainer.train()

In [ ]:
print(trainer.evaluate(metric_key_prefix="final"))

## 6. Inspect and save

In [ ]:
model.eval()
rows = [eval_ds[i] for i in range(4)]
inputs = processor([r["audio"]["array"] for r in rows], sampling_rate=SAMPLING_RATE,
                   padding="longest", return_tensors="pt").to(model.device, dtype=model.dtype)
with torch.no_grad():
    hyps = processor.batch_decode(model.generate(**inputs), skip_special_tokens=True)
for r, h in zip(rows, hyps):
    print(f"REF: {r['text'][:90]}\nHYP: {_basic(h)[:90]}\n")

In [ ]:
model.save_pretrained("granite-turboctc-de-final")
processor.save_pretrained("granite-turboctc-de-final")

## Notes

* **On a big GPU**, drop `gradient_checkpointing` and use `per_device_train_batch_size=32,
  gradient_accumulation_steps=1` — measured ~16x faster on an H100 at 21 GB peak. The defaults above
  fit a free Colab T4.
* **`learning_rate=1e-5`** suits a large domain shift like this one. If your baseline is already
  strong, it is too aggressive — start at `1e-6` with a longer warmup, or freeze the lower encoder:
  `for p in model.encoder.layers[:8].parameters(): p.requires_grad = False`.
* **More data helps most.** ~10 h against a 60k-hour pretrain is thin; expect a working pipeline
  rather than a production German model.